# Del 2: Vi trener vår egen språkmodell 🏗️

I Del 1 møtte du en **ferdigtrent** minimodell og så hvordan den genererer tekst, ett token om
gangen. Nå går vi bak kulissene og gjør jobben selv – **helt fra bunnen av**:

1. ✂️ Vi trener en **tokenizer** på *overordnet del av læreplanen*
2. 🔬 Vi ser **hvordan trening faktisk foregår** – hva «tap» er, og hvordan modellen justerer seg
3. 🏋️ Vi **trener modellen** og følger med mens den lærer (3–5 minutter)
4. 👀 Vi kikker inn i **attention** – mekanismen alle snakker om
5. 🎓 Vi **instruksjonstrener** modellen – slik at den går fra å *fortsette* tekst til å *svare* på spørsmål

> **Slik bruker du notebooken:** Kjør cellene ovenfra og ned – klikk ▶️ til venstre for cellen.
> Koden er skjult (cellene merket 🔧 er «maskinrommet» – bare trykk ▶️), og du trenger ikke
> forstå den. Alt kjører på vanlig CPU.


In [ ]:
#@title 🔧 Oppsett (kjør meg først) { display-mode: "form" }
import time, json, copy, warnings
from pathlib import Path

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import html as html_verktoy
from IPython.display import HTML, display

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import (GPT2Config, GPT2LMHeadModel, PreTrainedTokenizerFast,
                          pipeline, set_seed)
from transformers.utils import logging as hf_logging

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()
torch.manual_seed(42)

FARGER = ["#ffd6e7", "#d6eaff", "#d9f7d6", "#fff3c2", "#eadcff", "#ffe0c2", "#d6f5f2", "#e8e8e8"]

def vis_bokser(biter):
    bokser = "".join(
        f'<span style="background:{FARGER[i % len(FARGER)]}; padding:3px 2px; margin:2px; '
        f'border-radius:4px; font-family:monospace; font-size:16px; display:inline-block">'
        f'{html_verktoy.escape(str(b))}</span>'
        for i, b in enumerate(biter))
    display(HTML(f'<div style="line-height:2.4">{bokser}</div>'))

print("Klar! 🚀")

## Steg 1: Teksten som er alt modellen får 📚

En språkmodell lærer alt den kan fra **tekst** – ingenting annet. De store modellene trenes på
billioner av ord fra internett og bøker. Vår modell får **ett dokument**: *Overordnet del –
verdier og prinsipper for grunnopplæringen* (ca. 9 000 ord).


In [ ]:
#@title 🔧 Les inn teksten { display-mode: "form" }
FILNAVN = "overordnet_del.txt"
RAW_URL = "https://github.com/erlingmi/FIKS_kurs/blob/main/overordnet_del.txt"   # valgfritt: direktelenke til fila (f.eks. GitHub raw), så hentes den automatisk

if Path(FILNAVN).exists():
    text = Path(FILNAVN).read_text(encoding="utf-8")
elif RAW_URL:
    import urllib.request
    text = urllib.request.urlopen(RAW_URL).read().decode("utf-8")
else:
    try:
        from google.colab import files
        print(f"Fant ikke {FILNAVN} – last den opp her:")
        opplastet = files.upload()
        text = list(opplastet.values())[0].decode("utf-8")
    except ImportError:
        raise FileNotFoundError(f"Fant ikke {FILNAVN}. Legg fila i samme mappe som notebooken.")

print(f"Teksten er lest inn: {len(text):,} tegn, ca. {len(text.split()):,} ord.".replace(",", " "))

## Steg 2: Tokenizeren trenes ✂️

I Del 1 så du at modellen deler tekst i **tokens** (orddeler). Men hvor kommer oppdelingen fra?
Den **læres fra teksten**, med en forbløffende enkel oppskrift (**BPE** – *byte pair encoding*):

> Start med enkelttegn. Finn de to bitene som oftest står ved siden av hverandre, og lim dem
> sammen til én ny bit. Gjenta 800 ganger.

Ingen har fortalt tokenizeren noe om norsk – se hva den finner ut helt selv:


In [ ]:
#@title 🔧 Tren tokenizeren (tar noen sekunder) { display-mode: "form" }
tok = Tokenizer(models.BPE(unk_token="<ukjent>"))
tok.pre_tokenizer = pre_tokenizers.Metaspace(replacement="\u2581", prepend_scheme="always")
tok.decoder = decoders.Metaspace(replacement="\u2581", prepend_scheme="never")
tok.train_from_iterator([text], trainers.BpeTrainer(vocab_size=800,
                        special_tokens=["<ukjent>", "<eos>"]))

EOS = tok.token_to_id("<eos>")
hf_tok = PreTrainedTokenizerFast(tokenizer_object=tok, unk_token="<ukjent>",
                                 eos_token="<eos>", pad_token="<eos>")

merges = json.loads(tok.to_str())["model"]["merges"]
print("De 12 første sammenslåingene tokenizeren lærte helt selv:")
for a, b in merges[:12]:
    print(f"   {a!r} + {b!r}  →  {(a + b)!r}")
print()
print(f"Ferdig! Vokabular: {tok.get_vocab_size()} tokens.")
print("(ChatGPT og Claude har vokabular på 100 000–200 000 tokens – laget på nøyaktig samme måte.)")

Den fant `er`, `en` og `og` – de vanligste byggeklossene i norsk – uten å kunne noe som
helst om grammatikk. Vanlige ord i læreplanen blir til slutt **ett token**, sjeldne ord må
settes sammen av flere biter:


In [ ]:
#@title ✍️ Prøv tokenizeren { display-mode: "form" }
min_setning = "Elevene p\u00e5 Hellerud spiser taco hver fredag."  #@param {type:"string"}

enc = tok.encode(min_setning)
vis_bokser([t for t in enc.tokens])
print(f"→ {len(enc.tokens)} tokens, som for modellen bare er tall: {enc.ids}")

## Steg 3: Hva vil det si å «trene»? 🎯

Nå bygger vi selve modellen – et nevralt nettverk med ca. **420 000 justerbare tall**
(*parametere*). I starten er alle tallene tilfeldige: modellen kan **ingenting**.

Treningsoppskriften er den samme for alle språkmodeller, fra vår til ChatGPT:

> **Vis modellen en bit av teksten. Be den gjette neste token. Mål hvor mye den bommer.
> Juster parameterne bittelitt. Gjenta – millioner av ganger.**

Hele treningsløkka er faktisk bare disse fem linjene kode:

```python
for steg in range(800):
    x = hent_tekstbit()                  # en tilfeldig bit av læreplanen
    tap = modell(x).loss                 # hvor mye bommer modellen?
    tap.backward()                       # regn ut hvilken retning hvert tall bør justeres
    optimizer.step()                     # ... og juster alle 420 000 bittelitt
```

Det finnes ingen grammatikkregler og ingen fasit-ordbok. Bare: *her er teksten så langt – hva
kommer nå?* La oss se nøyaktig hva som skjer i **ett eneste** slikt treningssteg.


In [ ]:
#@title 🔧 Bygg modellen og hent ett treningseksempel { display-mode: "form" }
KONTEKST = 96   # hvor mange tokens bakover modellen kan se (kontekstvinduet)

# teksten deles i avsnitt, med <eos> («slutt») mellom – slik lærer modellen å avslutte
enheter = [a.strip() for a in text.split("\n\n") if a.strip()]
data_ids = []
for a in enheter:
    data_ids.extend(tok.encode(a).ids + [EOS])
data = torch.tensor(data_ids, dtype=torch.long)
n_val = int(0.1 * len(data))
tren_data, val_data = data[:-n_val], data[-n_val:]   # 10 % holdes utenfor treningen

def hent_batch(kilde, batch=32):
    ix = torch.randint(len(kilde) - KONTEKST - 1, (batch,))
    return torch.stack([kilde[i:i + KONTEKST] for i in ix])

config = GPT2Config(vocab_size=tok.get_vocab_size(), n_positions=KONTEKST,
                    n_embd=96, n_layer=3, n_head=4,
                    bos_token_id=None, eos_token_id=EOS, pad_token_id=EOS,
                    attn_implementation="eager")
modell = GPT2LMHeadModel(config)
antall_param = sum(p.numel() for p in modell.parameters())
print(f"Modellen er bygd: {antall_param:,} parametere – alle tilfeldige tall akkurat nå.".replace(",", " "))
print()

eksempel_tekst = "Skolen skal legge til rette for"
eksempel_ids = torch.tensor([tok.encode(eksempel_tekst).ids])
fasit_id = tok.token_to_id("\u2581l\u00e6ring") or tok.token_to_id("\u2581at")
print("Ett treningseksempel fra teksten:")
print(f'   KONTEKST (det modellen ser):  «{eksempel_tekst} …»')
print(f'   FASIT (tokenet den skal gjette): «{tok.id_to_token(fasit_id)}»')

## Steg 4: Tap og justering – trening på nært hold 🔬

Husker du søylediagrammet fra Del 1 – modellens sannsynlighet for hvert mulig neste token?
**Tapet** (*loss*) er rett og slett et mål på hvor **lav** sannsynlighet modellen ga
fasit-tokenet. Ga den fasiten 0,1 %, er tapet høyt. Ga den 90 %, er tapet lavt.

Og **justeringen** (*backpropagation*)? En algoritme regner ut, for hvert eneste av de 420 000
tallene i modellen: *«hvis akkurat dette tallet hadde vært bittelitt større eller mindre, ville
gjetningen blitt bedre eller verre?»* – og dytter så alle tallene et bitte lite hakk i riktig
retning.

La oss se det skje. Vi tar den splitter nye (utrente) modellen og trener den **10 steg på ett
eneste eksempel**, og følger med på søylediagrammet underveis:


In [ ]:
#@title 🔧 10 treningssteg i sakte film { display-mode: "form" }
demo = copy.deepcopy(modell)          # vi øver på en KOPI, så hovedmodellen forblir urørt
opt_demo = torch.optim.AdamW(demo.parameters(), lr=3e-3)

def fordeling(m):
    with torch.no_grad():
        logits = m(eksempel_ids).logits[0, -1]
    return F.softmax(logits, dim=-1)

fasit_logg, panel = [], {}
for steg in range(11):
    p = fordeling(demo)
    fasit_logg.append(p[fasit_id].item() * 100)
    if steg in (0, 3, 10):
        panel[steg] = p.clone()
    if steg < 10:
        fullt = torch.cat([eksempel_ids, torch.tensor([[fasit_id]])], dim=1)
        tap = demo(fullt, labels=fullt).loss
        opt_demo.zero_grad(); tap.backward(); opt_demo.step()

fig, akser = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
for ax, steg in zip(akser, panel):
    p = panel[steg]
    verdier, hvilke = torch.topk(p, 7)
    navn = [tok.id_to_token(i).replace("\u2581", " ") for i in hvilke.tolist()]
    farger = ["#dd4444" if i == fasit_id else "#4c72b0" for i in hvilke.tolist()]
    if fasit_id not in hvilke.tolist():   # fasit utenfor topp 7? vis den nederst i rødt
        navn.append(tok.id_to_token(fasit_id).replace("\u2581", " ") + "  (fasit)")
        verdier = torch.cat([verdier, p[fasit_id].unsqueeze(0)])
        farger.append("#dd4444")
    ax.barh([repr(n)[1:-1] for n in navn][::-1], (verdier * 100).tolist()[::-1],
            color=farger[::-1])
    ax.set_title(f"Etter {steg} steg")
    ax.set_xlabel("Sannsynlighet (%)")
    ax.grid(alpha=0.3, axis="x")
akser[0].figure.suptitle(f"Hva tror modellen kommer etter «{eksempel_tekst} …»?  (fasit i rødt)", y=1.05)
plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 3.5))
plt.plot(fasit_logg, marker="o", color="#dd4444")
plt.xlabel("Antall treningssteg"); plt.ylabel("Sannsynlighet for fasit (%)")
plt.title("Modellen blir sikrere på riktig svar for hvert steg")
plt.grid(alpha=0.3); plt.show()

Ser du? Fasit-tokenet starter med en sannsynlighet nær null (rød søyle), og for hvert
steg **klatrer det oppover lista** – fordi hver justering dytter de 420 000 tallene litt i
retning av riktig svar.

Og justeringene er virkelig *bitte små*. Her er tre av modellens 420 000 tall, før og etter
**ett** treningssteg:


In [ ]:
#@title 🔧 Tre av de 420 000 tallene, før og etter ett steg { display-mode: "form" }
demo2 = copy.deepcopy(modell)
w = demo2.transformer.h[0].attn.c_attn.weight
foer = w[0, :3].clone()
opt2 = torch.optim.AdamW(demo2.parameters(), lr=3e-3)
fullt = torch.cat([eksempel_ids, torch.tensor([[fasit_id]])], dim=1)
tap = demo2(fullt, labels=fullt).loss
opt2.zero_grad(); tap.backward(); opt2.step()
etter = w[0, :3]
print(f"{'FØR':>12}  {'ETTER':>12}  {'ENDRING':>12}")
for f, e in zip(foer.tolist(), etter.tolist()):
    print(f"{f:>12.6f}  {e:>12.6f}  {e - f:>+12.6f}")
print()
print("Trening = milliarder slike bittesmå dytt, fordelt på alle parameterne.")

## Steg 5: Full trening! 🏋️

Nå gjør vi det på ordentlig: **800 treningssteg**, hver med 32 tilfeldige tekstbiter fra
læreplanen. Underveis stopper vi opp og ber modellen skrive fritt fra «Skolen skal …», så vi
ser den utvikle seg. Vi tar også et «røntgenbilde» av attention-mekanismen ved hvert
stopp – det ser vi på i steg 7.

**Kjør cellen og følg med – dette tar 3–5 minutter.** ☕

*(Mens vi venter: Vår trening er 800 steg på én CPU. GPT-4-klassen trenes i månedsvis på titusenvis
av spesialbrikker, til en strømregning på flere hundre millioner kroner. Oppskriften er den samme.)*


In [ ]:
#@title 🔧 Tren modellen (3–5 min) { display-mode: "form" }
ANTALL_STEG = 800
opt = torch.optim.AdamW(modell.parameters(), lr=2e-3)

att_setning = "Elevene skal l\u00e6re \u00e5 samarbeide og utvikle evne til medbestemmelse"
att_ids = torch.tensor([tok.encode(att_setning).ids])

@torch.no_grad()
def mal_val_tap(batcher=6):
    modell.eval()
    s = sum(modell(x := hent_batch(val_data), labels=x).loss.item() for _ in range(batcher))
    modell.train()
    return s / batcher

@torch.no_grad()
def roentgen():
    modell.eval()
    att = modell(att_ids, output_attentions=True).attentions
    modell.train()
    return torch.stack([a[0] for a in att])   # (lag, hoder, T, T)

def skriv_fritt(seed):
    modell.eval()
    p = pipeline("text-generation", model=modell, tokenizer=hf_tok, device=-1)
    set_seed(seed)
    res = p("Skolen skal", max_new_tokens=32, do_sample=True, temperature=0.8,
            top_k=40, pad_token_id=EOS)
    modell.train()
    return res[0]["generated_text"].replace("\n", " ")

sjekkpunkter = [0, 100, 300, ANTALL_STEG]
tren_tap_logg, val_tap_logg, att_snapshots = [], [], {}

t0 = time.time()
for steg in range(ANTALL_STEG + 1):
    if steg in sjekkpunkter:
        att_snapshots[steg] = roentgen()
        print(f"—— Etter {steg} treningssteg ({time.time() - t0:.0f} s) skriver modellen: ——")
        print(f"   «{skriv_fritt(seed=steg + 1)}»")
        print()
    if steg == ANTALL_STEG:
        break
    x = hent_batch(tren_data)
    tap = modell(x, labels=x).loss
    opt.zero_grad(); tap.backward(); opt.step()
    tren_tap_logg.append(tap.item())
    if steg % 40 == 0:
        val_tap_logg.append((steg, mal_val_tap()))

print(f"Ferdig trent på {(time.time() - t0) / 60:.1f} minutter! 🎉")

Se på utviklingen: Først **rent tull**. Så norske småord (*og*, *skal*, *for*) – de er
vanligst, så de lønner seg å gjette. Så ordsalat med læreplan-gloser, og til slutt hele fraser
med nesten riktig grammatikk. **Ingen har lært modellen norsk** – alt kommer fra å gjette
neste orddel, om og om igjen.

## Steg 6: Tapskurven – og et viktig varsku 📉

Vi målte tapet underveis, både på teksten modellen trener på og på de 10 % vi **holdt skjult**:


In [ ]:
#@title 🔧 Vis tapskurven { display-mode: "form" }
def glatt(v, vindu=25):
    return [sum(v[max(0, i - vindu):i + 1]) / len(v[max(0, i - vindu):i + 1])
            for i in range(len(v))]

plt.figure(figsize=(9, 4.5))
plt.plot(glatt(tren_tap_logg), label="Tap på treningsteksten", linewidth=2)
plt.plot(*zip(*val_tap_logg), label="Tap på tekst modellen IKKE har sett", linewidth=2)
plt.xlabel("Antall treningssteg"); plt.ylabel("Tap (hvor mye modellen bommer)")
plt.title("Modellen blir bedre – men lærer den, eller pugger den?")
plt.legend(); plt.grid(alpha=0.3); plt.show()

- **Blå kurve synker hele veien:** stadig bedre på teksten den trener på.
- **Oransje kurve flater ut:** på *ny* tekst hjelper det ikke å trene mer. Modellen har begynt å
  **pugge treningsteksten utenat** i stedet for å lære generelle mønstre.

Dette kalles **overtilpasning**, og med ett eneste dokument er det uunngåelig. Det er derfor
ekte modeller trenes på enorme tekstmengder – og det er også derfor de av og til kan **gjengi
treningsdata ordrett**. (Kjenner du igjen debatten om opphavsrett og personvern? Dette er kjernen i den.)

## Steg 7: Et gløtt inn i attention 👀

Hvordan holder modellen styr på sammenhengen i en setning? Svaret er **attention**
(«oppmerksomhet»): Hvert token får «se tilbake» på alle de forrige og selv velge hvilke det vil
hente informasjon fra – og *hvordan* det velger, er noe modellen lærer under treningen.

Røntgenbildene vi tok underveis viser nettopp det. Hver rad er et token som gjetter; lyse felter
viser hvilke tidligere tokens det legger vekt på. (Trekanten øverst er tom fordi modellen aldri
får se **framover** – den gjetter jo det som kommer!)


In [ ]:
#@title 🔧 Attention: fra kaos til struktur { display-mode: "form" }
navn = [tok.id_to_token(i).replace("\u2581", " ") for i in att_ids[0].tolist()]
T = len(navn)

# Vi viser det hodet som har utviklet tydeligst struktur (mest vekt på nære tokens).
siste = att_snapshots[ANTALL_STEG]
poeng = {(l, h): sum(siste[l, h, i, i - 1].item() for i in range(1, T))
         for l in range(siste.shape[0]) for h in range(siste.shape[1])}
lag, hode = max(poeng, key=poeng.get)

fig, akser = plt.subplots(1, len(sjekkpunkter), figsize=(16, 4.2))
for ax, steg in zip(akser, sjekkpunkter):
    ax.imshow(att_snapshots[steg][lag, hode, :T, :T], cmap="viridis", vmin=0)
    ax.set_title(f"Etter {steg} steg")
    ax.set_xticks(range(T)); ax.set_xticklabels(navn, rotation=90, fontsize=7)
    ax.set_yticks(range(T)); ax.set_yticklabels(navn if ax is akser[0] else [], fontsize=7)
fig.suptitle(f"Ett av modellens 12 attention-hoder leser: «{att_setning}»", y=1.02)
plt.tight_layout(); plt.show()
print("Rad = tokenet som gjetter.  Lyse felter = tokens det henter informasjon fra.")

Før trening er oppmerksomheten **jevnt smurt utover** – modellen ser på alt og ingenting.
Gjennom treningen vokser det fram **struktur**: dette hodet har for eksempel lært å følge nøye
med på tokenene rett bak.

**Et ærlig forbehold:** Det er fristende å si at modellen «forstår» at ord hører sammen. Men vi
kan bare se *at* den vektlegger noe – ikke *hvorfor*. I store modeller med tusenvis av
attention-hoder er de aller fleste **ikke tolkbare**, heller ikke for forskerne som lager dem.
Når noen sier «ingen vet helt hva som skjer inni en språkmodell», er det blant annet dette de mener.


## Steg 8: Fra basemodell til assistent – instruksjonstrening 🎓

Nå har vi en **basemodell** – og i Del 1 så du problemet: still den et spørsmål, og den
*fortsetter* bare teksten i læreplanstil. Den svarer ikke. La oss bevise det igjen:


In [ ]:
#@title 🔧 Basemodellen får et spørsmål { display-mode: "form" }
def spor(m, sporsmal, seed=7, temperatur=0.4):
    p = pipeline("text-generation", model=m, tokenizer=hf_tok, device=-1)
    set_seed(seed)
    prompt = f"Sp\u00f8rsm\u00e5l: {sporsmal}\nSvar:"
    res = p(prompt, max_new_tokens=55, do_sample=True, temperature=temperatur, top_k=40,
            pad_token_id=EOS, eos_token_id=EOS)
    return res[0]["generated_text"][len(prompt):].strip().replace("\n", " ")

modell.eval()
for s in ["Hva er dybdel\u00e6ring?", "Hva er tilpasset oppl\u00e6ring?"]:
    print(f"❓ {s}")
    print(f"   «{spor(modell, s)}»")
    print()

Flytende læreplanprosa – men **ikke svar**. Løsningen heter **instruksjonstrening**
(*instruction tuning*), og den er overraskende udramatisk: Vi fortsetter nøyaktig samme
trening som før – gjett neste token! – men nå på tekst som ser slik ut:

```
Spørsmål: Hva er dybdelæring?
Svar: Dybdelæring er at elevene gradvis utvikler kunnskap og varig forståelse … <eos>
```

Når modellen har sett tusenvis av slike, har den lært et nytt *mønster*: etter «Spørsmål: …
Svar:» kommer det et svar. Det er slik ChatGPT og Claude ble assistenter – bare med millioner
av eksempler, håndskrevet og kvalitetssikret av mennesker.

Vi har laget **71 spørsmål–svar-par** om overordnet del. Her er tre av dem:


In [ ]:
#@title 🔧 Treningsdata: 71 spørsmål–svar-par { display-mode: "form" }
QA_PAR = [
    ('Hva bygger skolen sin praksis på?',
     'Skolen skal bygge sin praksis på verdiene i opplæringslovens formålsparagraf.'),
    ('Hva er grunnlaget for all opplæring?',
     'Menneskeverdets ukrenkelighet ligger til grunn for all opplæring og hele skolens virksomhet.'),
    ('Hvilke verdier bygger opplæringen på?',
     'Opplæringen bygger på menneskeverdet, identitet og kulturelt mangfold, kritisk tenkning, skaperglede, respekt for naturen og demokrati og medvirkning.'),
    ('Hva handler overordnet del om?',
     'Overordnet del beskriver verdiene og prinsippene som skal ligge til grunn for grunnopplæringen.'),
    ('Hva er formålet med opplæringen?',
     'Opplæringen skal gi elevene et godt grunnlag for å forstå seg selv, andre og verden, og for å gjøre gode valg i livet.'),
    ('Hva skal skolen gi elevene?',
     'Skolen skal gi elevene historisk og kulturell innsikt og forankring, og en god allmenndannelse.'),
    ('Hva vil det si at menneskeverdet er ukrenkelig?',
     'Alle mennesker har samme verdi uavhengig av bakgrunn og forutsetninger, og skolen skal legge dette til grunn for hele sin virksomhet.'),
    ('Hva skal skolen formidle?',
     'Skolen skal formidle kunnskap og fremme holdninger som sikrer demokratiet som samfunnsform.'),
    ('Hva sier overordnet del om identitet?',
     'Skolen skal bidra til at hver elev kan ivareta og utvikle sin identitet i et inkluderende og mangfoldig fellesskap.'),
    ('Hvorfor er kulturelt mangfold viktig i skolen?',
     'Et mangfoldig fellesskap gir elevene innsikt i ulike måter å tenke og leve på, og styrker deres egen identitet.'),
    ('Hva sier overordnet del om språk?',
     'Opplæringen skal sikre at elevene blir trygge språkbrukere og at de utvikler sin språklige identitet.'),
    ('Hvorfor er språk viktig?',
     'Språk gir oss tilhørighet og brukes til å tenke, skape mening, kommunisere og knytte bånd til andre.'),
    ('Hva er kritisk tenkning?',
     'Kritisk tenkning innebærer å vurdere ulike kilder til kunnskap og å tenke vitenskapelig, og bidrar til at elevene utvikler god dømmekraft.'),
    ('Hvorfor skal elevene lære kritisk tenkning?',
     'Kritisk tenkning og etisk bevissthet er en forutsetning for læring og bidrar til at elevene utvikler god dømmekraft.'),
    ('Hva er etisk bevissthet?',
     'Etisk bevissthet er å veie ulike hensyn mot hverandre, og er nødvendig for å være et reflektert menneske.'),
    ('Hva skal skolen gjøre med nysgjerrighet?',
     'Skolen skal bidra til at elevene blir nysgjerrige og stiller spørsmål, og utvikler vitenskapelig og kritisk tenkning.'),
    ('Hva er skaperglede?',
     'Skaperglede handler om at elevene skal få bruke sine skapende krefter og oppleve gleden ved å skape noe gjennom hele grunnopplæringen.'),
    ('Hva sier overordnet del om å prøve og feile?',
     'Elevene skal lære at det å prøve og feile er en del av det å lære.'),
    ('Hva er utforskertrang?',
     'Utforskertrang er lysten til å utforske og oppdage, og den er en del av det å lære.'),
    ('Hva utvikler elever gjennom skapende virksomhet?',
     'Elever som lærer gjennom skapende virksomhet, utvikler evnen til å uttrykke seg på ulike måter og til å løse problemer.'),
    ('Hva sier overordnet del om naturen?',
     'Skolen skal bidra til at elevene utvikler naturglede, respekt for naturen og klima- og miljøbevissthet.'),
    ('Hva skal elevene lære om klima?',
     'Elevene skal få innsikt i hvordan menneskets levesett påvirker naturen og klimaet, og hvordan vi kan leve mer bærekraftig.'),
    ('Hva er miljøbevissthet?',
     'Miljøbevissthet er å forstå hvordan våre valg påvirker naturen og klimaet, og å handle deretter.'),
    ('Hva sier overordnet del om demokrati?',
     'Skolen skal gi elevene mulighet til å medvirke og til å lære hva demokrati betyr i praksis.'),
    ('Hva betyr elevmedvirkning?',
     'Elevene skal erfare at de blir lyttet til i skolehverdagen, at de har reell innflytelse, og at de kan påvirke det som angår dem.'),
    ('Hvorfor er demokrati viktig i skolen?',
     'Skolen skal gjøre elevene i stand til å delta i demokratiske prosesser og styrke deres demokratiske beredskap.'),
    ('Hva skal elevene lære om medbestemmelse?',
     'Elevene skal lære å samarbeide og utvikle evne til medbestemmelse og medansvar.'),
    ('Hva er sosial læring?',
     'Sosial læring skjer både i undervisningen og i alle andre aktiviteter i skolens regi, og kan ikke isoleres fra faglig læring.'),
    ('Hvordan blir elevenes identitet til?',
     'Elevenes identitet og selvbilde, meninger og holdninger blir til i samspill med andre.'),
    ('Hva sier overordnet del om vennskap?',
     'Vennskap skaper tilhørighet og gjør oss alle mindre sårbare.'),
    ('Hva er grunnlaget for empati?',
     'Å kunne sette seg inn i hva andre tenker, føler og erfarer, er grunnlaget for empati og vennskap.'),
    ('Hva er kompetanse?',
     'Kompetanse er å kunne tilegne seg og anvende kunnskaper og ferdigheter til å mestre utfordringer og løse oppgaver i kjente og ukjente sammenhenger.'),
    ('Hva innebærer kompetanse?',
     'Kompetanse innebærer forståelse og evne til refleksjon og kritisk tenkning.'),
    ('Hva er de grunnleggende ferdighetene?',
     'De grunnleggende ferdighetene er å kunne lese, skrive, regne og ha muntlige og digitale ferdigheter.'),
    ('Hvorfor er grunnleggende ferdigheter viktige?',
     'De er en forutsetning for å kunne delta i utdanning, arbeid og samfunnsliv.'),
    ('Hva er dybdelæring?',
     'Dybdelæring er at elevene gradvis utvikler kunnskap og varig forståelse av begreper, metoder og sammenhenger i og mellom fag.'),
    ('Hva krever dybdelæring?',
     'Dybdelæring krever at elevene reflekterer over egen læring og bruker det de har lært i kjente og ukjente sammenhenger.'),
    ('Hva skal skolen gi rom for?',
     'Skolen skal gi rom for dybdelæring slik at elevene utvikler forståelse av sentrale elementer og sammenhenger innenfor et fag.'),
    ('Hva vil det si å lære å lære?',
     'Å lære å lære handler om at elevene reflekterer over sin egen læring og blir aktive deltakere i egne læringsprosesser.'),
    ('Hvorfor skal elevene reflektere over egen læring?',
     'Elever som forstår sine egne læringsprosesser, kan tilegne seg kunnskap på selvstendig vis.'),
    ('Hva er de tverrfaglige temaene?',
     'De tre tverrfaglige temaene er folkehelse og livsmestring, demokrati og medborgerskap, og bærekraftig utvikling.'),
    ('Hva er folkehelse og livsmestring?',
     'Temaet skal gi elevene kompetanse som fremmer god psykisk og fysisk helse, og som gir muligheter til å ta ansvarlige livsvalg.'),
    ('Hva er demokrati og medborgerskap?',
     'Temaet skal gi elevene kunnskap om demokratiets forutsetninger, verdier og spilleregler.'),
    ('Hva er bærekraftig utvikling som tema?',
     'Temaet skal legge til rette for at elevene kan forstå grunnleggende dilemmaer og utviklingstrekk i samfunnet.'),
    ('Hva kjennetegner et godt læringsmiljø?',
     'Et raust og støttende læringsmiljø er grunnlaget for en positiv kultur der elevene oppmuntres til faglig og sosial utvikling.'),
    ('Hva skal skolen utvikle?',
     'Skolen skal utvikle inkluderende fellesskap som fremmer helse, trivsel og læring for alle.'),
    ('Hva slags utfordringer skal elevene møte?',
     'Elevene skal møte utfordringer de kan vokse på, som de kan mestre på egen hånd eller sammen med andre.'),
    ('Hva er tilpasset opplæring?',
     'Tilpasset opplæring er at alle elever får best mulig utbytte av opplæringen uavhengig av forutsetninger og bakgrunn, innenfor fellesskapet.'),
    ('Hvordan skal undervisningen tilpasses?',
     'Undervisningen skal tilpasses den enkelte elev og elevgruppens forutsetninger.'),
    ('Hvor skjer tilpasset opplæring?',
     'Tilpasset opplæring skjer innenfor fellesskapet.'),
    ('Hvem har hovedansvaret for barnas oppdragelse?',
     'Foreldrene og foresatte har hovedansvaret for barnas oppdragelse og utvikling.'),
    ('Hva sier overordnet del om samarbeid med hjemmet?',
     'Skolen skal samarbeide godt med hjemmet, og samarbeidet skal bygge på gjensidig respekt og tillit.'),
    ('Hva er et profesjonsfellesskap?',
     'Skolen skal være et profesjonsfaglig fellesskap der lærere, ledere og andre ansatte reflekterer over felles verdier og videreutvikler sin praksis.'),
    ('Hva krever god skoleutvikling?',
     'God skoleutvikling krever rom for å stille spørsmål og lete etter svar, og et profesjonsfellesskap som engasjerer seg i skolens utvikling.'),
    ('Hva skal skolens ledelse gjøre?',
     'Skolens ledelse skal gi retning for og tilrettelegge for elevenes og lærernes læring og utvikling.'),
    ('Hvordan utvikler lærere god praksis?',
     'Lærere som i fellesskap reflekterer over og vurderer undervisningen, utvikler en rikere forståelse av god pedagogisk praksis.'),
    ('Hvor blir skolens formål realisert?',
     'Det er gjennom det daglige møtet mellom elever og lærere at skolens brede formål blir realisert.'),
    ('Hvilke avveininger må lærere gjøre?',
     'Lærere må hele tiden gjøre krevende avveininger mellom hensynet til den enkelte elev og hensynet til fellesskapet.'),
    ('Hva er elevens beste?',
     'Alle elever er ulike, og hva som er elevens beste, er et kjernespørsmål som må besvares på nytt hver dag av alle som jobber i skolen.'),
    ('Hva sier overordnet del om motivasjon?',
     'Skolen skal stimulere den enkeltes motivasjon, lærelyst og tro på egen mestring.'),
    ('Hva skal elevene utvikle?',
     'Elevene skal utvikle kunnskap, ferdigheter og holdninger for å kunne mestre livene sine og delta i arbeid og fellesskap i samfunnet.'),
    ('Kan du forklare hva dybdelæring er?',
     'Dybdelæring er at elevene gradvis utvikler kunnskap og varig forståelse av begreper, metoder og sammenhenger i og mellom fag.'),
    ('Forklar tilpasset opplæring.',
     'Tilpasset opplæring er at alle elever får best mulig utbytte av opplæringen uavhengig av forutsetninger og bakgrunn, innenfor fellesskapet.'),
    ('Fortell om de tverrfaglige temaene.',
     'De tre tverrfaglige temaene er folkehelse og livsmestring, demokrati og medborgerskap, og bærekraftig utvikling.'),
    ('Kan du forklare hva kompetanse er?',
     'Kompetanse er å kunne tilegne seg og anvende kunnskaper og ferdigheter til å mestre utfordringer og løse oppgaver i kjente og ukjente sammenhenger.'),
    ('Forklar hva kritisk tenkning er.',
     'Kritisk tenkning innebærer å vurdere ulike kilder til kunnskap og å tenke vitenskapelig, og bidrar til at elevene utvikler god dømmekraft.'),
    ('Fortell om elevmedvirkning.',
     'Elevene skal erfare at de blir lyttet til i skolehverdagen, at de har reell innflytelse, og at de kan påvirke det som angår dem.'),
    ('Hva menes med grunnleggende ferdigheter?',
     'De grunnleggende ferdighetene er å kunne lese, skrive, regne og ha muntlige og digitale ferdigheter.'),
    ('Hva menes med sosial læring?',
     'Sosial læring skjer både i undervisningen og i alle andre aktiviteter i skolens regi, og kan ikke isoleres fra faglig læring.'),
    ('Hva menes med skaperglede?',
     'Skaperglede handler om at elevene skal få bruke sine skapende krefter og oppleve gleden ved å skape noe.'),
    ('Fortell om samarbeidet mellom hjem og skole.',
     'Skolen skal samarbeide godt med hjemmet, og samarbeidet skal bygge på gjensidig respekt og tillit.'),
]

import random as tilfeldig
tilfeldig.seed(3)
for sp, sv in tilfeldig.sample(QA_PAR, 3):
    print(f"Sp\u00f8rsm\u00e5l: {sp}")
    print(f"Svar: {sv}")
    print()
print(f"... og {len(QA_PAR) - 3} til.")

In [ ]:
#@title 🔧 Instruksjonstren modellen (ca. 1 min) { display-mode: "form" }
qa_ids = [hf_tok(f"Sp\u00f8rsm\u00e5l: {sp}\nSvar: {sv}")["input_ids"] + [EOS]
          for sp, sv in QA_PAR]

def qa_batch(batch=16):
    valg = [qa_ids[i] for i in torch.randint(len(qa_ids), (batch,))]
    L = max(len(s) for s in valg)
    x = torch.full((batch, L), EOS, dtype=torch.long)
    fasit = torch.full((batch, L), -100, dtype=torch.long)   # -100 = «tell ikke med»
    maske = torch.zeros((batch, L), dtype=torch.long)
    for r, s in enumerate(valg):
        x[r, :len(s)] = torch.tensor(s)
        fasit[r, :len(s)] = torch.tensor(s)
        maske[r, :len(s)] = 1
    return x, fasit, maske

assistent = copy.deepcopy(modell)      # vi beholder basemodellen og trener videre på en kopi
assistent.train()
opt_a = torch.optim.AdamW(assistent.parameters(), lr=1e-3)

t0 = time.time()
for steg in range(300):
    x, fasit, maske = qa_batch()
    tap = assistent(x, labels=fasit, attention_mask=maske).loss
    opt_a.zero_grad(); tap.backward(); opt_a.step()
    if steg % 100 == 0:
        print(f"steg {steg}: tap {tap.item():.2f}")
assistent.eval()
print(f"Ferdig på {(time.time() - t0) / 60:.1f} min – samme oppskrift, bare nye data.")

Sannhetens øyeblikk. Samme spørsmål som i stad – først til den nye assistenten. Og så
noen spørsmål den **umulig kan vite noe om**:


In [ ]:
#@title 🔧 Før og etter – og en felle { display-mode: "form" }
print("=" * 60)
print("SPØRSMÅL MODELLEN HAR TRENT PÅ:")
print("=" * 60)
for s in ["Hva er dybdel\u00e6ring?", "Hva er tilpasset oppl\u00e6ring?"]:
    print(f"❓ {s}")
    print(f"   Basemodellen:  «{spor(modell, s)}»")
    print(f"   Assistenten:   «{spor(assistent, s)}»")
    print()
print("=" * 60)
print("SPØRSMÅL UTENFOR ALT DEN HAR SETT:")
print("=" * 60)
for s in ["Hva er en god matpakke?", "Hvem vant VM i fotball i 2022?"]:
    print(f"❓ {s}")
    print(f"   Assistenten:   «{spor(assistent, s)}»")
    print()

To ting skjedde – og begge er blant de viktigste innsiktene i hele kurset:

1. **Assistenten svarer.** Den har lært *formatet* «spørsmål → svar». Det er hele forskjellen
   på en basemodell og en chatbot.
2. **Men den svarer også når den ikke kan.** Spør du om matpakker eller fotball-VM, får du et
   selvsikkert, velformulert – og fullstendig irrelevant – svar. Den har **ingen mekanisme for å
   vite at den ikke vet**. Den fullfører bare mønsteret.

Dette er **hallusinasjon**, sett innenfra: Instruksjonstreningen lærte modellen å *høres ut som*
en assistent. Kunnskapen fulgte ikke med på kjøpet. Store modeller demper dette med enorme
kunnskapsmengder og egen trening i å si «det vet jeg ikke» – men mekanismen under er den samme,
og det er derfor selv de beste modellene kan servere feil med full selvtillit.


In [ ]:
#@title ✍️ Still ditt eget spørsmål til assistenten { display-mode: "form" }
mitt_sporsmal = "Hva er de tverrfaglige temaene?"  #@param {type:"string"}
temperatur = 0.4  #@param {type:"slider", min:0.1, max:1.5, step:0.1}

print(f"❓ {mitt_sporsmal}")
print(f"   «{spor(assistent, mitt_sporsmal, seed=int(time.time()), temperatur=float(temperatur))}»")
print()
print("Tips: prøv både spørsmål fra læreplanens verden og spørsmål helt utenfor!")

---

## Det store bildet 🗺️

Du har nå gjort – i miniatyr – nøyaktig det OpenAI og Anthropic gjør:

| | Vår modell | De store |
|---|---|---|
| Tokenizer | BPE, 800 tokens | BPE, 100 000–200 000 tokens |
| Treningsdata | 1 dokument (~9 000 ord) | Billioner av ord |
| Parametere | 420 000 | Mange milliarder |
| Trening | 800 steg, 4 min, én CPU | Måneder, titusenvis av GPU-er |
| Instruksjonstrening | 71 syntetiske par | Millioner av menneskelagde eksempler |
| + i tillegg | – | RLHF: mennesker rangerer svar, modellen justeres mot det de foretrekker |

**Grunnmuren er identisk**: tokens → gjett neste → mål tapet → juster → gjenta. Alt annet er skala.

## Lek videre – valgfrie eksperimenter 🧪

Cellene nedenfor er til å utforske på egen hånd (eller hvis det er tid til overs):


In [ ]:
#@title 🧪 Valgfritt: Tren basemodellen videre { display-mode: "form" }
# Blir teksten bedre – eller begynner modellen bare å sitere læreplanen ordrett?
# (Sammenlign med originalteksten! Og se hva som skjer med den oransje kurven om du
#  kjører tapskurve-cellen på nytt etterpå.)
EKSTRA_STEG = 800  #@param {type:"slider", min:200, max:2000, step:200}

modell.train()
t0 = time.time()
for steg in range(EKSTRA_STEG):
    x = hent_batch(tren_data)
    tap = modell(x, labels=x).loss
    opt.zero_grad(); tap.backward(); opt.step()
    tren_tap_logg.append(tap.item())
    if steg % 40 == 0:
        val_tap_logg.append((ANTALL_STEG + steg, mal_val_tap()))
modell.eval()
print(f"Trente {EKSTRA_STEG} ekstra steg på {(time.time()-t0)/60:.1f} min. Slik skriver den nå:")
print(f"   «{skriv_fritt(seed=999)}»")

In [ ]:
#@title 🧪 Valgfritt: Finetuning på en helt annen tekst { display-mode: "form" }
# Dette er «vanlig» finetuning: fortsett treningen på et nytt dokument, og se stilen endre seg.
# Last opp en tekstfil (eventyr, avisartikler, skolens ordensreglement – minst noen tusen ord)
# og kjør cellen. NB: Tokenizeren er trent på læreplanen, så helt nye ord blir delt i småbiter.
try:
    from google.colab import files
    print("Last opp en .txt-fil:")
    ny = list(files.upload().values())[0].decode("utf-8")
except ImportError:
    ny = Path("annen_tekst.txt").read_text(encoding="utf-8")   # utenfor Colab: legg fila her

ny_ids = []
for a in ny.split("\n\n"):
    if a.strip():
        ny_ids.extend(tok.encode(a.strip()).ids + [EOS])
ny_data = torch.tensor(ny_ids, dtype=torch.long)
print(f"Ny tekst: {len(ny_ids):,} tokens. Finetuner 400 steg …".replace(",", " "))

fin = copy.deepcopy(modell)
fin.train()
opt_f = torch.optim.AdamW(fin.parameters(), lr=1e-3)
for steg in range(400):
    x = hent_batch(ny_data)
    tap = fin(x, labels=x).loss
    opt_f.zero_grad(); tap.backward(); opt_f.step()
fin.eval()
p = pipeline("text-generation", model=fin, tokenizer=hf_tok, device=-1)
set_seed(1)
res = p("Det var en gang", max_new_tokens=40, do_sample=True, temperature=0.8, top_k=40,
        pad_token_id=EOS)
print(f"   «{res[0]['generated_text']}»")

**Flere ideer:**

- **Mindre vokabular:** Sett `vocab_size=100` i tokenizer-cellen og kjør alt på nytt. Hvordan
  ser tokenene ut nå? Hva skjer med modellens tekst?
- **Ødelegg assistenten:** Kjør instruksjonstrenings-cellen med bare 30 steg i stedet for 300.
  Hvor mye format har den lært da?
- **Kontekstvinduet:** Modellen ser maks 96 tokens bakover (`KONTEKST = 96`). Det er derfor
  språkmodeller «glemmer» starten av lange samtaler – konteksten er alltid endelig
  (ChatGPT og Claude: fra hundretusen tokens og oppover).

---

*Notebooken hører til Del 2 av kurset «Språkmodeller for skoleledere og lærere». Del 1 finner du i
`del1_demo.ipynb`. Modellen fra Del 1 ligger på Hugging Face: `erlingmi/mini-laereplan-gpt`.*
